# Aula 6 – Agrupando e Resumindo com Groupby

## 1. Objetivos da aula
Ao final desta aula, você será capaz de:

* Usar `groupby` para criar resumos por categoria (ex.: por cidade, por disciplina, por produto).
* Aplicar funções como `sum`, `mean`, `count`, `max`, `min` em grupos.
* Usar `agg` para calcular múltiplas métricas ao mesmo tempo.
* Agrupar por uma ou mais colunas.
* Criar tabelas tipo tabela dinâmica com `pivot_table`.

Essa é a aula que transforma tabela “crua” em relatórios de negócio.

## 2. Por que usar `groupby`?
Situações típicas do dia a dia:

* Quanto faturamos por cidade?
* Qual é a média de nota por disciplina?
* Quantas vendas foram feitas por canal de venda?
* Qual é o ticket médio por cliente?

Você não quer calcular isso na mão. Você quer dizer para o pandas: “Agrupe por X e me traga o resumo.”

`groupby` é o coração dessa operação.

## 3. Dataset exemplo: Vendas por cidade e canal
Vamos criar um DataFrame de vendas:

In [ ]:
import pandas as pd

dados_vendas = {
    "id_venda": [1, 2, 3, 4, 5, 6, 7, 8],
    "cidade": [
        "Juiz de Fora", "Juiz de Fora", "São Paulo", "São Paulo",
        "Rio de Janeiro", "Rio de Janeiro", "Juiz de Fora", "São Paulo"
    ],
    "canal_venda": [
        "Online", "Loja Física", "Online", "Marketplace",
        "Loja Física", "Online", "Online", "Loja Física"
    ],
    "valor": [200.0, 350.0, 500.0, 300.0, 800.0, 150.0, 400.0, 250.0]
}

df_vendas = pd.DataFrame(dados_vendas)
df_vendas

## 4. Agrupando por uma coluna

### 4.1. Soma do valor por cidade
Queremos o faturamento total por cidade:

In [ ]:
faturamento_por_cidade = df_vendas.groupby("cidade")["valor"].sum()
faturamento_por_cidade

Isso devolve uma Series com:
* índice = cidades
* valor = soma do valor

Se quiser em formato de DataFrame:

In [ ]:
faturamento_por_cidade = df_vendas.groupby("cidade", as_index=False)["valor"].sum()
faturamento_por_cidade

Aqui, `as_index=False` faz a coluna cidade continuar como coluna normal, não vira índice.

### 4.2. Outras funções: média, quantidade, mínimo, máximo

In [ ]:
df_vendas.groupby("cidade")["valor"].mean()   # média por cidade

In [ ]:
df_vendas.groupby("cidade")["valor"].count()  # quantidade de vendas

In [ ]:
df_vendas.groupby("cidade")["valor"].max()    # maior venda em cada cidade

## 5. Agrupando por mais de uma coluna
Agora queremos ver faturamento por cidade E canal de venda.

In [ ]:
agrupado = df_vendas.groupby(["cidade", "canal_venda"])["valor"].sum()
agrupado

Isso retorna uma Series com índice multi-nível. Se você quiser como DataFrame:

In [ ]:
agrupado_df = df_vendas.groupby(["cidade", "canal_venda"], as_index=False)["valor"].sum()
agrupado_df

## 6. Usando `agg` para múltiplas métricas
O `agg` permite calcular várias estatísticas de uma vez.

Exemplo: valor total, médio e número de vendas por cidade:

In [ ]:
resumo_cidade = df_vendas.groupby("cidade")["valor"].agg(["sum", "mean", "count"])
resumo_cidade

Você pode renomear as colunas:

In [ ]:
resumo_cidade = df_vendas.groupby("cidade")["valor"].agg(
    total_faturado="sum",
    ticket_medio="mean",
    qtde_vendas="count"
)
resumo_cidade

Se quiser tirar o índice e transformar cidade em coluna de novo:

In [ ]:
resumo_cidade = resumo_cidade.reset_index()

## 7. Agrupando múltiplas colunas com `agg` 
Imagine que queremos um resumo por cidade com soma de valor, número de vendas (`id_venda`) e a venda máxima. Você pode passar um dicionário para o `agg`:

In [ ]:
resumo = df_vendas.groupby("cidade").agg({
    "valor": ["sum", "mean", "max"],
    "id_venda": "count"
})
resumo

Isso cria um DataFrame com colunas multi-nível. Se quiser algo mais legível, pode agregar em uma coluna só ou renomear como vimos anteriormente.

## 8. Pivot Table – Tabela dinâmica em Pandas
`pivot_table` é uma forma mais “direta” de criar tabelas estilo Excel (tabela dinâmica).

Exemplo: queremos uma tabela com linhas = cidades, colunas = canais de venda e valores = soma do valor.

In [ ]:
tabela = pd.pivot_table(
    df_vendas,
    values="valor",
    index="cidade",
    columns="canal_venda",
    aggfunc="sum",
    fill_value=0
)

tabela

Parâmetros importantes:
* `values` → qual coluna será agregada
* `index` → o que será linha
* `columns` → o que será coluna
* `aggfunc` → qual função usar ("sum", "mean", "count", etc.)
* `fill_value` → o que colocar quando não houver valor (ex.: 0)

## 9. Dataset exemplo 2: Notas por disciplina
Agora vamos para um contexto de notas de alunos por disciplina:

In [ ]:
dados_notas = {
    "aluno": [
        "Ana", "Ana", "Ana",
        "Bruno", "Bruno", "Carla", "Carla", "Daniel", "Eduarda", "Eduarda"
    ],
    "disciplina": [
        "Matemática", "Português", "História",
        "Matemática", "História", "Matemática", "Português", "História", "Matemática", "Português"
    ],
    "nota": [8.0, 7.5, 9.0, 6.0, 7.0, 9.5, 8.0, 5.5, 7.0, 8.5]
}

df_notas = pd.DataFrame(dados_notas)
df_notas

### 9.1. Média de notas por disciplina

In [ ]:
media_por_disciplina = df_notas.groupby("disciplina")["nota"].mean()
media_por_disciplina

Se quiser como DataFrame, com coluna renomeada:

In [ ]:
media_por_disciplina = (
    df_notas.groupby("disciplina", as_index=False)["nota"]
            .mean()
            .rename(columns={"nota": "media_nota"})
)
media_por_disciplina

### 9.2. Média de notas por aluno

In [ ]:
media_por_aluno = (
    df_notas.groupby("aluno", as_index=False)["nota"]
            .mean()
            .rename(columns={"nota": "media_nota"})
)
media_por_aluno

Podemos ordenar para ver o ranking:

In [ ]:
media_por_aluno.sort_values("media_nota", ascending=False)

### 9.3. Resumo geral por disciplina (média, min, max)

In [ ]:
resumo_disciplina = df_notas.groupby("disciplina")["nota"].agg(
    media="mean",
    nota_min="min",
    nota_max="max",
    qtde_registros="count"
)
resumo_disciplina

### 9.4. Tabela tipo “aluno x disciplina”
Com `pivot_table`, podemos organizar linhas = alunos, colunas = disciplinas e valores = nota.

In [ ]:
tabela_notas = pd.pivot_table(
    df_notas,
    values="nota",
    index="aluno",
    columns="disciplina",
    aggfunc="mean"
)

tabela_notas

## 10. Resumo da aula
Hoje você aprendeu a:

* **Usar `groupby` para agrupar por:** uma coluna ou mais de uma coluna.
* **Aplicar funções** como `sum`, `mean`, `count`, `max`, `min` em grupos.
* **Usar `agg`** para calcular múltiplas métricas em uma única chamada.
* **Criar tabelas estilo “tabela dinâmica”** com `pd.pivot_table`.

Essa é uma das ferramentas mais importantes de quem trabalha com dados: pegar um monte de linhas e transformá-las em resumos que fazem sentido pro negócio.